# generate_lens_queries

**Objective:** Build batched Lens.org patent queries for the startup sample, one cohort per founding year, applying the priority-date window and the European residence filter.

**Inputs:** `../02_matching_crunchbase_data/startups_master.csv`.

**Outputs:** `lens_queries/batch_*.txt` query files plus `_batch_mapping.csv` and `_batch_summary.csv`.

## Imports

In [ ]:
# Imports
from pathlib import Path
import re
import pandas as pd

## Paths and config

In [ ]:
# Paths and configuration
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
INPUT_PATH = PROC / "startups_master.csv"
OUTPUT_DIR = PROC / "lens_queries"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_PRE  = 3
N_POST = 10
MAX_QUERY_CHARS = 4000

## Load startups and derive founding year + country

In [ ]:
# Load startups; derive founding year and country
df = pd.read_csv(INPUT_PATH).drop_duplicates("Organization Name URL")
df["founded_year"] = pd.to_numeric(
    df["Founded Date"].astype(str).str.extract(r"(\d{4})", expand=False),
    errors="coerce",
).astype("Int64")
df["_country"] = df["Headquarters Location"].astype(str).str.split(", ").str[-1]

print(f"Sample: {len(df)} unique companies")
print(f"Founded-Year-Range: {df['founded_year'].min()} - {df['founded_year'].max()}")

## Map countries to ISO codes and drop unmapped

In [ ]:
# Country-name to ISO-code map; drop unmapped firms
COUNTRY_TO_ISO = {
    "United Kingdom": "GB", "France": "FR", "Sweden": "SE", "Germany": "DE",
    "Switzerland": "CH", "Spain": "ES", "Italy": "IT", "The Netherlands": "NL",
    "Finland": "FI", "Ireland": "IE", "Denmark": "DK", "Norway": "NO",
    "Belgium": "BE", "Austria": "AT", "Poland": "PL", "Portugal": "PT",
    "Iceland": "IS", "Luxembourg": "LU", "Lithuania": "LT", "Estonia": "EE",
    "Latvia": "LV", "Czech Republic": "CZ", "Czechia": "CZ", "Slovakia": "SK",
    "Slovakia (Slovak Republic)": "SK", "Hungary": "HU", "Romania": "RO",
    "Bulgaria": "BG", "Slovenia": "SI", "Croatia": "HR", "Greece": "GR",
    "Cyprus": "CY", "Malta": "MT", "Liechtenstein": "LI",
    "Russian Federation": "RU", "Ukraine": "UA", "Turkey": "TR",
}
df["_iso"] = df["_country"].map(COUNTRY_TO_ISO)
unmapped = df[df["_iso"].isna()]
if len(unmapped):
    print(f"WARN: {len(unmapped)} companies in unmapped countries")
    print(unmapped["_country"].value_counts())
    df = df.dropna(subset=["_iso"])
all_iso = sorted(df["_iso"].unique())
print(f"Final: {len(df)} companies, {len(all_iso)} countries")

## Clean company names (strip legal suffixes)

In [ ]:
# Legal-suffix list for company-name cleaning
LEGAL_SUFFIXES = [
    "gmbh", "ag", "sa", "sas", "sarl", "srl", "sl", "bv", "nv", "ab", "oy",
    "as", "asa", "aps", "plc", "llc", "ltd", "limited", "inc", "corp",
    "corporation", "llp", "holding", "holdings", "group", "the",
    "co", "company", "international", "intl", "systems", "technologies",
    "tech", "solutions",
]

def clean_name(name):
    n = str(name).lower().strip()
    n = re.sub(r"[^\w\s&+]", " ", n)
    n = re.sub(r"\s+", " ", n).strip()
    parts = n.split()
    while parts and parts[-1] in LEGAL_SUFFIXES:
        parts.pop()
    return " ".join(parts)

df["_clean_name"] = df["Organization Name"].apply(clean_name)
df = df[df["_clean_name"] != ""]
df = df.dropna(subset=["founded_year"])
print(f"After cleaning: {len(df)} companies with Founded-Year")

## Query builder helpers and budget calculation

In [ ]:
# Jurisdiction list and query-budget helpers
JURISDICTIONS = ["EP", "WO", "DE", "FR", "GB", "IT", "ES", "NL", "SE",
                 "DK", "FI", "NO", "CH", "AT", "BE", "IE", "PL", "PT"]

def name_to_term(clean):
    return f'applicant.name:("{clean}")' if " " in clean else f'applicant.name:{clean}'

def build_query(names, iso_codes, date_from, date_to):
    name_part = " OR ".join(name_to_term(n) for n in names)
    res_part  = " OR ".join(iso_codes)
    return (
        f"({name_part})"
        f" AND earliest_priority_claim_date:[{date_from} TO {date_to}]"
        f" AND applicant.residence:({res_part})"
        f" AND has_abstract:true"
    )

OVERHEAD = len(build_query(["x"], all_iso, "2000-01-01", "2030-12-31")) - len("applicant.name:x")
BUDGET = MAX_QUERY_CHARS - OVERHEAD
print(f"Overhead per query: {OVERHEAD} chars")
print(f"Budget for company names: {BUDGET} chars")

## Clear previous batch outputs

In [ ]:
# Clear previous batch query files
for old in OUTPUT_DIR.glob("batch_*.txt"):
    old.unlink()
for old in OUTPUT_DIR.glob("_batch_mapping*.csv"):
    old.unlink()

## Build per-cohort batches and write query files

In [ ]:
# Build per-cohort query batches and write files
mapping_rows = []
batch_summary = []

for year in sorted(df["founded_year"].dropna().unique()):
    year = int(year)
    cohort = df[df["founded_year"] == year]
    date_from = f"{year - N_PRE}-01-01"
    date_to   = f"{year + N_POST}-12-31"

    records = cohort[["Organization Name", "Organization Name URL",
                       "_clean_name", "_country", "_iso"]].to_dict("records")
    sub_batches = []
    cur, cur_chars = [], 0
    for rec in records:
        term = name_to_term(rec["_clean_name"])
        add  = len(term) + (4 if cur else 0)
        if cur_chars + add > BUDGET and cur:
            sub_batches.append(cur)
            cur, cur_chars = [], 0
            add = len(term)
        cur.append(rec)
        cur_chars += add
    if cur:
        sub_batches.append(cur)

    for i, batch in enumerate(sub_batches, 1):
        batch_id = f"batch_{year}_{i:02d}"
        names    = [r["_clean_name"] for r in batch]
        query    = build_query(names, all_iso, date_from, date_to)
        (OUTPUT_DIR / f"{batch_id}.txt").write_text(query, encoding="utf-8")

        batch_summary.append({
            "batch": batch_id, "founded_year": year,
            "date_from": date_from, "date_to": date_to,
            "n_companies": len(batch), "query_chars": len(query),
        })
        for r in batch:
            mapping_rows.append({
                "batch": batch_id, "founded_year": year,
                "organization_name": r["Organization Name"],
                "organization_url":  r["Organization Name URL"],
                "clean_name_for_lens": r["_clean_name"],
                "country": r["_country"], "iso": r["_iso"],
            })

## Write mapping/summary CSVs and report stats

In [ ]:
# Write mapping and summary CSVs; report statistics
summary_df = pd.DataFrame(batch_summary)
pd.DataFrame(mapping_rows).to_csv(OUTPUT_DIR / "_batch_mapping.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "_batch_summary.csv", index=False)

print(f"{len(summary_df)} batches generated")
print(f"Query lengths: min={summary_df['query_chars'].min()}, "
      f"max={summary_df['query_chars'].max()}, "
      f"avg={summary_df['query_chars'].mean():.0f}")

by_year = summary_df.groupby("founded_year").agg(
    n_batches=("batch", "size"),
    n_companies=("n_companies", "sum"),
    date_range=("date_from", lambda x: f"{x.iloc[0]} -> {summary_df.loc[x.index[0], 'date_to']}"),
).reset_index()
print(by_year.to_string(index=False))

## Preview the first generated batch file

In [ ]:
# Preview the first generated batch file
first_batch_file = sorted(OUTPUT_DIR.glob("batch_*.txt"))[0]
print(f"File: {first_batch_file.name}")
print(f"Length: {len(first_batch_file.read_text())} chars")
print(first_batch_file.read_text()[:600] + "  ...")